# NeuralMath — Η μαθηματική ανατομία ενός νευρωνικού δικτύου

Παρουσιάζουμε μια μαθηματική εισαγωγή στα τεχνητά νευρωνικά δίκτυα μέσα από ένα απλό πρόβλημα ταξινόμησης. Ένας γραμμικός ταξινομητής οδηγεί φυσικά στην έννοια του τεχνητού νευρώνα· στη συνέχεια συζητούμε τα βάρη, τους όρους πόλωσης, τις συναρτήσεις ενεργοποίησης και τη σύνθεση νευρώνων σε ένα μικρό δίκτυο. Η εκπαίδευση εισάγεται μέσω συναρτήσεων απώλειας, παραγώγων, καθόδου κλίσης και οπισθοδιάδοσης. Τέλος, υλοποιούμε το μοντέλο απευθείας σε Python, ώστε η αντιστοιχία ανάμεσα στις εξισώσεις και τον αλγόριθμο να είναι ρητά ορατή. Στόχος είναι να δείξουμε πώς οικείες ιδέες από την αναλυτική γεωμετρία, τη γραμμική άλγεβρα και τον διαφορικό λογισμό παρέχουν μια προσιτή βάση για την κατανόηση των θεμελιωδών μηχανισμών των νευρωνικών δικτύων.

**NeuralMath.org** · *A anatomia matemática de uma rede neural*

Τα ονόματα μεταβλητών και συναρτήσεων διατηρούνται στα αγγλικά· το επεξηγηματικό κείμενο είναι στη γλώσσα της σελίδας.


## 1. Κατασκευή ενός μικρού νευρωνικού δικτύου

Τα ονόματα μεταβλητών και συναρτήσεων διατηρούνται στα αγγλικά· το επεξηγηματικό κείμενο είναι στη γλώσσα της σελίδας.

$$\mathbf{z}^{(1)}=W^{(1)}\mathbf{x}+\mathbf{b}^{(1)},\qquad \mathbf{h}=\operatorname{ReLU}(\mathbf{z}^{(1)}),$$

$$z^{(2)}=W^{(2)}\mathbf{h}+b^{(2)},\qquad \widehat y=\sigma(z^{(2)}).$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def relu(s):
    return np.maximum(0, s)


def relu_derivative(s):
    return (s > 0).astype(float)


def sigmoid(s):
    return 1 / (1 + np.exp(-s))


def base_boundary(m):
    return (
        6.68
        + 0.74 * np.exp(-((m - 155) / 20) ** 2)
        + 1.50 / (1 + np.exp(-(m - 202) / 5))
    )


## 2. Μαθηματική αναπαράσταση του προβλήματος

Για να κατανοήσουμε πώς λειτουργεί ένα νευρωνικό δίκτυο, ξεκινάμε με ένα σκόπιμα απλό πρόβλημα: να διακρίνουμε μήλα από πορτοκάλια. Ένα φρούτο διαθέτει πολλά χαρακτηριστικά που θα μπορούσαν να είναι χρήσιμα για αυτόν τον σκοπό, όπως χρώμα, μάζα, διάμετρος, υφή και άρωμα. Ωστόσο, ένα πρώτο μαθηματικό μοντέλο δεν χρειάζεται να περιλαμβάνει όλη αυτή την πολυπλοκότητα. Θα χρησιμοποιήσουμε μόνο δύο εύκολα μετρήσιμες ποσότητες: τη μάζα, σε γραμμάρια, και τη διάμετρο, σε εκατοστά.

$$x_j^{\ast}=\frac{x_j-\mu_j}{\sigma_j}.$$


In [ ]:
def build_dataset():
    masses_1 = np.linspace(118, 212, 16)
    masses_2 = masses_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7,
    ])
    k = np.arange(16)

    apple_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) + 0.22 + 0.05 * np.sin(0.8 * k),
    ])
    apple_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) + 0.58 + 0.06 * np.cos(0.45 * k),
    ])
    orange_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) - (0.22 + 0.04 * np.cos(0.7 * k)),
    ])
    orange_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) - (0.55 + 0.05 * np.sin(0.5 * k)),
    ])

    X_train = np.vstack([apple_1, apple_2, orange_1, orange_2])
    y_train = np.array([1.0] * 32 + [0.0] * 32)

    X_test = np.array([
        [121.0, 7.20], [135.0, 7.45], [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86], [189.0, 6.48], [208.0, 7.55],
    ])
    y_test = np.array([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])
    return X_train, y_train, X_test, y_test


X_train, y_train, X_test, y_test = build_dataset()
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
Xn = (X_train - mean) / std

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('mean:', mean)
print('std:', std)


## 3. Πώς μαθαίνει ένα νευρωνικό δίκτυο;

Τα βάρη και οι όροι πόλωσης δεν είναι γνωστά εκ των προτέρων. Πρέπει να προσαρμοστούν χρησιμοποιώντας παραδείγματα για τα οποία είναι διαθέσιμη η σωστή απάντηση. Η διαδικασία αυτή ονομάζεται εκπαίδευση και μπορεί να συνοψιστεί στον κύκλο \[\boxed{\text{πρόβλεψε}}
\longrightarrow
\boxed{\text{μέτρησε το σφάλμα}}
\longrightarrow
\boxed{\text{προσάρμοσε τις παραμέτρους}}
\longrightarrow
\boxed{\text{επανάλαβε}}.\]


In [ ]:
rng = np.random.default_rng(42)
W1 = 0.1 * rng.standard_normal((12, 2))
b1 = np.zeros(12)
W2 = 0.1 * rng.standard_normal(12)
b2 = 0.0

learning_rate = 0.05
n_epochs = 6000
history = []

for epoch in range(n_epochs):
    for i in rng.permutation(len(Xn)):
        x = Xn[i]
        target = y_train[i]

        z1 = W1 @ x + b1
        h = relu(z1)
        z2 = W2 @ h + b2
        y_hat = sigmoid(z2)

        delta2 = (y_hat - target) * y_hat * (1 - y_hat)
        dW2 = delta2 * h
        db2 = delta2

        delta1 = (delta2 * W2) * relu_derivative(z1)
        dW1 = np.outer(delta1, x)
        db1 = delta1

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

    errors = []
    for x, target in zip(Xn, y_train):
        h = relu(W1 @ x + b1)
        y_hat = sigmoid(W2 @ h + b2)
        errors.append(0.5 * (target - y_hat) ** 2)
    history.append(float(np.mean(errors)))

print(f"E_final: {history[-1]:.10f}")


## 4. Από τα μαθηματικά στην Python

Περνάμε τώρα από τις εξισώσεις σε ένα εκτελέσιμο πρόγραμμα. Ο στόχος δεν είναι να χρησιμοποιήσουμε μια εξειδικευμένη βιβλιοθήκη νευρωνικών δικτύων, αλλά να υλοποιήσουμε απευθείας με το NumPy τις μαθηματικές πράξεις που αναπτύχθηκαν σε αυτό το άρθρο . Έτσι γίνεται ορατό, γραμμή προς γραμμή, πώς οι γραμμικοί συνδυασμοί, οι συναρτήσεις ενεργοποίησης, ο υπολογισμός της απώλειας, η οπισθοδιάδοση και η κάθοδος κλίσης εμφανίζονται στον κώδικα.


In [ ]:
def predict(X):
    X_scaled = (X - mean) / std
    outputs = []
    for x in X_scaled:
        h = relu(W1 @ x + b1)
        outputs.append(sigmoid(W2 @ h + b2))
    return np.array(outputs)


pred_train = predict(X_train)
pred_test = predict(X_test)

print(f"acc_train: {np.mean((pred_train >= 0.5) == y_train):.4f}")
print(f"acc_test: {np.mean((pred_test >= 0.5) == y_test):.4f}")
print('y_hat_test:', np.round(pred_test, 6))

masses = np.linspace(116, 214, 500)
diameters = np.linspace(6.0, 8.85, 500)
M, D = np.meshgrid(masses, diameters)
grid = np.column_stack([M.ravel(), D.ravel()])
P = predict(grid).reshape(M.shape)

fig, ax = plt.subplots(figsize=(9.2, 6.3))
ax.contourf(
    M, D, P,
    levels=[0.0, 0.25, 0.50, 0.75, 1.0],
    colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
    alpha=0.95,
)
ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'], linewidths=2.2)

mask = y_train == 1.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='o', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='Μήλο — εκπαίδευση')
mask = y_train == 0.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='s', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='Πορτοκάλι — εκπαίδευση')
mask = y_test == 1.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='^', s=92,
           color='#1f77b4', label='Μήλο — δοκιμή')
mask = y_test == 0.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='x', s=110,
           color='#ff7f0e', linewidths=2.0, label='Πορτοκάλι — δοκιμή')

ax.set_xlim(116, 214)
ax.set_ylim(6.0, 8.85)
ax.set_xlabel('Μάζα (g)')
ax.set_ylabel('Διάμετρος (cm)')
ax.set_title('Έξοδος του νευρωνικού δικτύου μετά την εκπαίδευση', fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)
fig.tight_layout()
plt.show()


## 5. NeuralMath

Όλο το υπολογιστικό υλικό που συνοδεύει αυτό το άρθρο — συμπεριλαμβανομένου του κώδικα και ενός εκτελέσιμου σημειωματάριου Google Colab — είναι διαθέσιμο στη διεύθυνση https://neuralmath.org , ως ενιαίο σημείο πρόσβασης για αναγνώστες που θέλουν να εξερευνήσουν, να αναπαράγουν ή να προσαρμόσουν τα παραδείγματα.
